# Zero-Shot Model Analysis (Thesis Section 4.3)

Compares zero-shot performance of three open-source LLMs on the full dataset (2,550 responses):
- **Gemma 2 9B-IT**
- **Llama 3 8B-Instruct**
- **DeepSeek 7B-Chat**

**Sections:**
1. Load Results
2. Overall Metrics Comparison
3. Grade Distribution (Human vs Model)
4. Per-Band Accuracy
5. Per-Rubric Exact Accuracy
6. Per-Rubric Per-Score-Level Heatmaps
7. Off-Topic Detection Analysis
8. Model Selection Justification

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score, confusion_matrix

sns.set_theme(style='whitegrid', font_scale=1.1)

BASE = '../Zeroshot'
MODELS = {
    'Gemma 2 9B-IT': 'results_Gemma2-9B-IT_full_dataset.csv',
    'Llama 3 8B-Instruct': 'results_Llama3-8B-Instruct_full_dataset.csv',
    'DeepSeek 7B-Chat': 'results_DeepSeek-7B-Chat_full_dataset.csv'
}
RUBRICS = ['Clarity', 'Terminology', 'Coverage', 'Accuracy']

## 1. Load Results

In [ ]:
data = {}
for name, filename in MODELS.items():
    df = pd.read_csv(f'{BASE}/{filename}')
    # Normalize column names
    col_map = {}
    for c in df.columns:
        if c in ('Within_1', 'Within_1.0'): col_map[c] = 'Within_1'
        elif c in ('Within_05', 'Within_0.5'): col_map[c] = 'Within_05'
    df = df.rename(columns=col_map)
    data[name] = df
    print(f'{name}: {len(df)} samples loaded')

# Show sample
data['Gemma 2 9B-IT'].head(3)

## 2. Overall Metrics Comparison

In [ ]:
def compute_metrics(df):
    """Compute all evaluation metrics."""
    diff = np.abs(df['LLM_Final'].values - df['Human_Final'].values)
    acc_1 = np.mean(diff <= 1.0) * 100
    acc_05 = np.mean(diff <= 0.5) * 100
    mae = np.mean(diff)

    # QWK on total rubric score
    h_total = (df['Human_Clarity'] + df['Human_Terminology'] + df['Human_Coverage'] + df['Human_Accuracy']).astype(int)
    l_total = (df['LLM_Clarity'] + df['LLM_Terminology'] + df['LLM_Coverage'] + df['LLM_Accuracy']).astype(int).clip(0, 10)
    qwk = cohen_kappa_score(h_total, l_total, weights='quadratic')

    return {'Acc ±1.0 (%)': acc_1, 'Acc ±0.5 (%)': acc_05, 'MAE': mae, 'QWK': qwk}


metrics_rows = []
for name, df in data.items():
    m = compute_metrics(df)
    m['Model'] = name
    metrics_rows.append(m)

metrics_df = pd.DataFrame(metrics_rows)[['Model', 'Acc ±1.0 (%)', 'Acc ±0.5 (%)', 'MAE', 'QWK']]
metrics_df = metrics_df.round(2)
print('Overall Zero-Shot Performance (2,550 samples):')
metrics_df

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
model_names = list(data.keys())
colors = ['#2ecc71', '#3498db', '#e74c3c']

for ax, metric in zip(axes, ['Acc ±1.0 (%)', 'Acc ±0.5 (%)', 'MAE', 'QWK']):
    values = [compute_metrics(data[m])[metric] for m in model_names]
    bars = ax.bar(range(len(model_names)), values, color=colors, edgecolor='white')
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([n.split()[0] for n in model_names], fontsize=10)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    for bar, val in zip(bars, values):
        fmt = f'{val:.1f}%' if '%' in metric else f'{val:.3f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01 * max(values),
                fmt, ha='center', va='bottom', fontsize=10)

fig.suptitle('Zero-Shot Model Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Grade Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, df) in zip(axes, data.items()):
    bins = np.arange(-0.25, 5.5, 0.5)
    ax.hist(df['Human_Final'], bins=bins, alpha=0.6, label='Human', color='#2c3e50', edgecolor='white')
    ax.hist(df['LLM_Final'], bins=bins, alpha=0.6, label=name.split()[0], color='#e74c3c', edgecolor='white')
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Final Grade (0-5)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

fig.suptitle('Grade Distribution: Human vs Zero-Shot Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Per-Band Accuracy (Macro-Averaged)

In [ ]:
def per_band_accuracy(df, tolerance=1.0):
    """Compute accuracy within tolerance for each human grade band."""
    bands = sorted(df['Human_Final'].unique())
    results = {}
    for band in bands:
        mask = df['Human_Final'] == band
        if mask.sum() > 0:
            diff = np.abs(df.loc[mask, 'LLM_Final'] - df.loc[mask, 'Human_Final'])
            results[band] = np.mean(diff <= tolerance) * 100
    return results


band_data = {}
for name, df in data.items():
    band_data[name] = per_band_accuracy(df)

bands_df = pd.DataFrame(band_data).round(1)
bands_df.index.name = 'Human Grade'
print('Per-Band ±1.0 Accuracy (%):')
bands_df

In [ ]:
# Per-band accuracy plot
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(bands_df.index))
width = 0.25

for i, (name, color) in enumerate(zip(model_names, colors)):
    vals = [band_data[name].get(b, 0) for b in bands_df.index]
    ax.bar(x + i * width, vals, width, label=name.split()[0], color=color, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels([str(b) for b in bands_df.index])
ax.set_xlabel('Human Grade Band', fontsize=12)
ax.set_ylabel('±1.0 Accuracy (%)', fontsize=12)
ax.set_title('Per-Band Accuracy Comparison', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## 5. Per-Rubric Exact Accuracy

In [ ]:
def per_rubric_accuracy(df):
    """Compute exact match accuracy per rubric criterion."""
    results = {}
    for rubric in RUBRICS:
        human_col = f'Human_{rubric}'
        llm_col = f'LLM_{rubric}'
        exact = np.mean(df[human_col] == df[llm_col]) * 100
        within_1 = np.mean(np.abs(df[human_col] - df[llm_col]) <= 1) * 100
        mae = np.mean(np.abs(df[human_col] - df[llm_col]))
        results[rubric] = {'Exact (%)': round(exact, 1), '±1 (%)': round(within_1, 1), 'MAE': round(mae, 3)}
    return pd.DataFrame(results).T


for name, df in data.items():
    print(f'\n{name}:')
    display(per_rubric_accuracy(df))

## 6. Per-Rubric Per-Score-Level Heatmaps

In [ ]:
for name, df in data.items():
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.suptitle(f'{name} — Confusion Matrices by Rubric', fontsize=14, y=1.05)

    for ax, rubric in zip(axes, RUBRICS):
        human_col = f'Human_{rubric}'
        llm_col = f'LLM_{rubric}'
        max_score = 4 if rubric == 'Accuracy' else 2
        labels = list(range(max_score + 1))

        cm = confusion_matrix(df[human_col], df[llm_col], labels=labels)
        # Normalize by row (human score)
        cm_norm = cm.astype(float)
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_norm = np.divide(cm_norm, row_sums, where=row_sums > 0) * 100

        sns.heatmap(cm_norm, annot=True, fmt='.0f', cmap='Blues',
                    xticklabels=labels, yticklabels=labels, ax=ax,
                    vmin=0, vmax=100, cbar=False)
        ax.set_title(f'{rubric} (0-{max_score})', fontsize=11)
        ax.set_xlabel('LLM Score')
        ax.set_ylabel('Human Score')

    plt.tight_layout()
    plt.show()

## 7. Off-Topic Detection Analysis

In [ ]:
def off_topic_analysis(df, name):
    """Analyze off-topic detection performance."""
    # Determine off-topic column name variants
    if 'Human_Off_Topic' in df.columns:
        h_col, l_col = 'Human_Off_Topic', 'LLM_Off_Topic'
        # Boolean format: True = off-topic
        human_ot = df[h_col].astype(bool)
        llm_ot = df[l_col].astype(bool)
    else:
        h_col, l_col = 'Human_OffTopic', 'LLM_OffTopic'
        human_ot = df[h_col].isin(['Off-Topic', 'No Answer'])
        llm_ot = df[l_col].isin(['Off-Topic', 'No Answer'])

    tp = ((human_ot) & (llm_ot)).sum()
    fp = ((~human_ot) & (llm_ot)).sum()
    fn = ((human_ot) & (~llm_ot)).sum()
    tn = ((~human_ot) & (~llm_ot)).sum()

    total_ot = human_ot.sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    overall_acc = (tp + tn) / len(df) * 100

    return {
        'Model': name, 'Total Off-Topic': int(total_ot),
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'Precision': round(precision, 3), 'Recall': round(recall, 3),
        'F1': round(f1, 3), 'Overall Acc (%)': round(overall_acc, 1)
    }


ot_rows = [off_topic_analysis(df, name) for name, df in data.items()]
ot_df = pd.DataFrame(ot_rows)
print('Off-Topic Detection Performance:')
ot_df

## 8. Model Selection Justification

In [ ]:
# Summary comparison
print('='*60)
print('MODEL SELECTION SUMMARY')
print('='*60)
print()
print('Gemma 2 9B-IT was selected for fine-tuning based on:')
print()

for metric in ['Acc ±1.0 (%)', 'Acc ±0.5 (%)', 'MAE', 'QWK']:
    vals = {name: compute_metrics(df)[metric] for name, df in data.items()}
    if metric == 'MAE':
        best = min(vals, key=vals.get)
    else:
        best = max(vals, key=vals.get)
    marker = ' <-- BEST' if 'Gemma' in best else ''
    print(f'  {metric}:')
    for name, val in vals.items():
        is_best = ' *' if name == best else ''
        fmt = f'{val:.1f}%' if '%' in metric else f'{val:.4f}'
        print(f'    {name:<25} {fmt}{is_best}')
    print()

print('Key advantages of Gemma 2 9B-IT:')
print('  1. Highest ±1.0 accuracy across all models')
print('  2. Best QWK (strongest agreement with human graders)')
print('  3. Lowest MAE (most precise grading)')
print('  4. Best zero-shot off-topic detection consistency')